# Railway Headway & Capacity Simulator
## Google Colab Application Launcher

**Software Version:** `0.1.0-dev`  
**Current Phase:** `P00 — Project Foundation`  
**Interface:** Gradio Blocks  
**Execution Mode:** Google Colab (Inline Display)

---

### Instructions for Running in Google Colab:
1. **Step 1:** Verify runtime compatibility (Python >= 3.10 required).
2. **Step 2:** Ensure the `railway-headway-simulator` repository is present and installed.
3. **Step 3:** Initialize runtime directories and logging.
4. **Step 4:** Launch the interactive Gradio interface directly in the cell below.

*Note: Per Governance Directive RHS-MASTER-001, all engineering calculations execute in the Python package in the background. End users do not need to edit code.*


In [ ]:
# Step 1: Verify Python Runtime Compatibility
import sys
import platform

print("=== Google Colab Runtime Verification ===")
print(f"Python Version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")

if sys.version_info < (3, 10):
    raise RuntimeError(
        f"Unsupported Python version {platform.python_version()}. "
        "Railway Headway Simulator requires Python 3.10 or newer."
    )
print("✔ Runtime environment compatibility confirmed.")


In [ ]:
# Step 2: Install Package and Dependencies
import os
import subprocess
from pathlib import Path

# Verify if we are inside the repository or need to install
REPO_DIR = Path.cwd()

# If running in a fresh Colab instance where the repo was uploaded as a ZIP or cloned:
# Example: !git clone <USER_SUPPLIED_GIT_URL>
# Or unzip uploaded archive:
# !unzip -q railway-headway-simulator.zip -d .

print(f"Working Directory: {REPO_DIR}")

# Install package in editable/local mode
print("Installing/verifying railway-headway-simulator package...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    capture_output=True,
    text=True,
    cwd=REPO_DIR
)

if result.returncode != 0:
    print("Standard installation failed. Output:")
    print(result.stderr)
    raise RuntimeError("Failed to install package. Please ensure the repository root contains pyproject.toml.")

print("✔ Package successfully installed.")


In [ ]:
# Step 3: Initialize Configuration, Logging, and Directories
from headway.core.configuration import AppConfig, EnvironmentType
from headway.core.logging_config import setup_logging, get_logger
from headway.core.version import __version__, get_build_info

# Initialize Colab-optimized configuration
config = AppConfig(
    environment=EnvironmentType.COLAB,
    log_level="INFO",
    dev_mode=True
)
config.initialize_runtime()

# Initialize centralized logging
setup_logging(
    level=config.log_level,
    log_dir=config.directories.logs_dir
)

logger = get_logger("colab.launcher")
logger.info("Initializing Railway Headway Simulator v%s for Colab", __version__)
print(f"✔ Runtime initialized. Build info: {get_build_info()}")
print(f"✔ Storage directories ready at: {config.directories.base_dir}")


In [ ]:
# Step 4: Launch Gradio Interface
import gradio as gr
from headway.ui.app import create_app

# Close any existing active Gradio servers to prevent port collisions on re-run
try:
    gr.close_all()
except Exception:
    pass

# Create the Gradio interface
demo = create_app(config)

# Launch with inline Colab display
# Note: share=False by default to protect proprietary engineering datasets
demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False,
    inline=True,
    show_error=True,
    theme=getattr(demo, "app_theme", None),
    css=getattr(demo, "app_css", None),
)


### Step 5: Troubleshooting & Operation Notes

- **Restarting the UI:** If you need to reload the UI after modifying configuration, re-run Cell 4 and Cell 5. The launcher automatically executes `gr.close_all()` to release port 7860.
- **Port Conflict:** If port 7860 is occupied by another process, change `server_port=7861` in `config.server`.
- **Public Share Link:** If remote browser access outside Colab is strictly needed and permitted, set `config.server.share = True` before launching.
- **Runtime Persistence:** Files stored in Google Colab's local filesystem are temporary and destroyed when the runtime disconnects. Ensure all project packages and exports are downloaded to your local computer.
